## 1. Setup and Environment

Initialize the Tushare Pro API connection and import shared config (data directory paths, API token).

In [ ]:
import sys
sys.path.append(r"D:\Quant\Multi_factor_project")

from config import *
import tushare as ts
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime

pro = ts.pro_api(TUSHARE_TOKEN)
print("Tushare connected successfully, version:", ts.__version__)

In [ ]:
monthly_universe = pd.read_parquet(
    os.path.join(DATA_RAW, 'industry', 'hs300_monthly_universe.parquet')
)
all_stocks = monthly_universe['con_code'].unique().tolist()
print(f"Loaded successfully, {len(all_stocks)} stocks in total")

## 2. Download HS300 Historical Constituents

Fetch CSI 300 (HS300) index constituents **month by month** rather than in a single call, to avoid the silent 6000-row truncation on `index_weight`. This is essential for downloading the *full historical* constituent list without survivorship bias.

In [ ]:
def get_hs300_history_members(start='20150101', end='20241231'):
    """Fetch CSI 300 (HS300) index constituents month by month, to avoid silent
    truncation caused by the 6000-row limit on a single API call."""
    months = pd.date_range(start, end, freq='MS')
    frames = []
    for m in months:
        ms = m.strftime('%Y%m%d')
        me = (m + pd.offsets.MonthEnd(0)).strftime('%Y%m%d')
        for attempt in range(3):
            try:
                df = pro.index_weight(index_code='399300.SZ',
                                      start_date=ms, end_date=me)
                if df is not None and len(df) > 0:
                    frames.append(df)
                break
            except Exception as e:
                time.sleep((attempt + 1) * 2)
        time.sleep(0.35)
    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates(subset=['trade_date', 'con_code'])
    print(f"Raw constituent records: {len(out)}")   # should be around 36,000
    return out.sort_values(['trade_date', 'con_code']).reset_index(drop=True)

members_raw = get_hs300_history_members()
members_raw.to_parquet(
    os.path.join(DATA_RAW, 'index', 'hs300_members_raw.parquet')  # renamed the folder along the way
)

In [ ]:
os.makedirs(os.path.join(DATA_RAW, 'index'), exist_ok=True)
members_raw.to_parquet(os.path.join(DATA_RAW, 'index', 'hs300_members_raw.parquet'))
print("Save complete")

## 3. Build Month-End Universe Table

Collapse the raw weight records into one constituent snapshot per stock per month, and extract the full set of tickers that have ever been in the index — this defines the download universe for the next steps (not just current constituents).

In [ ]:
def build_monthly_universe(members_raw):
    """
    Reorganize the raw weight data into a month-end constituent list.
    trade_date format: YYYYMMDD
    """
    df = members_raw.copy()
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    
    # Take the constituents as of the last trading day of each month
    df['month'] = df['trade_date'].dt.to_period('M')
    
    monthly = (
        df.sort_values('trade_date')
          .groupby(['month', 'con_code'])
          .last()
          .reset_index()
    )
    
    # Sanity check: number of constituents per month
    count_per_month = monthly.groupby('month')['con_code'].count()
    print("Number of constituents per month (should be close to 300):")
    print(count_per_month.tail(12))
    
    return monthly

monthly_universe = build_monthly_universe(members_raw)
monthly_universe.to_parquet(
    os.path.join(DATA_RAW, 'industry', 'hs300_monthly_universe.parquet')
)

# Get all stock codes that have ever appeared in the index (full historical universe)
all_stocks = monthly_universe['con_code'].unique().tolist()
print(f"\n{len(all_stocks)} stocks have appeared in the index historically")

## 4. Download Daily Price Data

Download unadjusted daily OHLCV data for every ticker in the historical universe. Price adjustment (rights/dividends) is applied separately in Section 8 via the adjustment factor.

In [ ]:
def download_single_stock(ts_code, start_date, end_date, max_retries=3):
    """Download a single stock's data, with retry logic."""
    for attempt in range(max_retries):
        try:
            df = pro.daily(
                ts_code=ts_code,
                start_date=start_date,
                end_date=end_date,
                fields='ts_code,trade_date,open,high,low,close,vol,amount,pct_chg'
            )
            if df is not None and len(df) > 0:
                return df
            return None
        except Exception as e:
            wait = (attempt + 1) * 2  # wait 2s on 1st retry, 4s on 2nd, 6s on 3rd
            print(f"  Retry {attempt+1}/{max_retries}: {ts_code}, waiting {wait}s, reason: {e}")
            time.sleep(wait)
    return None  # give up only after 3 failed attempts

def download_daily_data(stock_list, start_date='20150101', end_date='20241231'):
    save_dir = os.path.join(DATA_RAW, 'daily')
    os.makedirs(save_dir, exist_ok=True)
    
    failed = []
    skipped = 0

    for i, ts_code in enumerate(stock_list):
        save_path = os.path.join(save_dir, f"{ts_code}.parquet")
        if os.path.exists(save_path):
            skipped += 1
            continue
        
        df = download_single_stock(ts_code, start_date, end_date)
        
        if df is not None:
            df['trade_date'] = pd.to_datetime(df['trade_date'])
            df = df.sort_values('trade_date').reset_index(drop=True)
            df.to_parquet(save_path)
        else:
            print(f"Final failure: {ts_code}")
            failed.append(ts_code)
        
        if (i + 1) % 20 == 0:
            print(f"Progress: {i+1}/{len(stock_list)}, skipped {skipped}, failed {len(failed)}")
        
        time.sleep(0.8)  # more conservative than the previous 0.3, to reduce timeouts

    print(f"\nDone: succeeded {len(stock_list)-len(failed)-skipped}, skipped {skipped}, failed {len(failed)}")
    if failed:
        print(f"Failed list: {failed}")
    return failed

print(f"Preparing to download {len(all_stocks)} stocks (already-downloaded ones are skipped automatically)...")
failed_stocks = download_daily_data(all_stocks)

## 5. Download Benchmark Index Data

Download CSI 300 index daily data, used later as the backtest benchmark for Alpha/Beta regression.

In [ ]:
def download_index_daily():
    """Download CSI 300 index daily data, to be used as the backtest benchmark."""
    df = pro.index_daily(
        ts_code='399300.SZ',
        start_date='20150101',
        end_date='20241231'
    )
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    df = df.sort_values('trade_date').reset_index(drop=True)
    
    save_path = os.path.join(DATA_RAW, 'daily', 'hs300_index.parquet')
    df.to_parquet(save_path)
    print(f"CSI 300 index download complete, {len(df)} trading days")
    print(df.tail(3))

download_index_daily()

## 6. Download Industry Classification

Fetch the Shenwan Level-1 (2021 edition) industry classification, used for industry-neutral factor standardization and portfolio construction.

In [ ]:
def download_industry_classification():
    """
    Fetch the Shenwan Level-1 industry classification (2021 edition).
    Returns the industry each stock belongs to.
    """
    # Get the list of Shenwan Level-1 industries
    industry_list = pro.index_classify(level='L1', src='SW2021')
    print("Shenwan Level-1 industries ({} in total):".format(len(industry_list)))
    print(industry_list[['index_code', 'industry_name']])
    
    # Get the constituents of each industry
    all_members = []
    for _, row in industry_list.iterrows():
        members = pro.index_member(index_code=row['index_code'])
        members['industry_name'] = row['industry_name']
        members['industry_code'] = row['index_code']
        all_members.append(members)
        time.sleep(0.5)
    
    industry_map = pd.concat(all_members, ignore_index=True)
    
    save_path = os.path.join(DATA_RAW, 'industry', 'sw_industry_map.parquet')
    industry_map.to_parquet(save_path)
    
    print(f"\nIndustry mapping download complete, {len(industry_map)} records")
    print(industry_map.head())
    
    return industry_map

industry_map = download_industry_classification()

In [ ]:
import pandas as pd
import os

monthly_universe = pd.read_parquet(
    os.path.join(DATA_RAW, 'industry', 'hs300_monthly_universe.parquet')
)
all_stocks = monthly_universe['con_code'].unique().tolist()

print(f"Loaded successfully, {len(all_stocks)} stocks in total")

## 7. Download Financial Statement Data

Download income statement, balance sheet, cash flow, and daily valuation data (PE/PB/market cap) for all tickers, incrementally backfilling only tickers not already saved.

In [ ]:
def patch_financial_data(stock_list, start_date='20150101', end_date='20241231'):
    save_dir = os.path.join(DATA_RAW, 'financial')
    apis = {
        'income': pro.income,
        'balancesheet': pro.balancesheet,
        'cashflow': pro.cashflow,
        'daily_basic': pro.daily_basic,
    }
    for name, api in apis.items():
        path = os.path.join(save_dir, f'{name}.parquet')
        old = pd.read_parquet(path)
        have = set(old['ts_code'].unique())
        missing = [c for c in stock_list if c not in have]
        print(f"{name}: {len(have)} already present, {len(missing)} missing")
        if not missing:
            continue

        frames = [old]
        for i, code in enumerate(missing):
            for attempt in range(3):
                try:
                    kwargs = dict(ts_code=code, start_date=start_date, end_date=end_date)
                    if name == 'daily_basic':
                        kwargs['fields'] = 'ts_code,trade_date,pe,pe_ttm,pb,total_mv,circ_mv'
                    df = api(**kwargs)
                    if df is not None and len(df) > 0:
                        frames.append(df)
                    break
                except Exception:
                    time.sleep((attempt + 1) * 2)
            if (i + 1) % 50 == 0:
                print(f"  {name}: {i+1}/{len(missing)}")
            time.sleep(0.5)

        merged = pd.concat(frames, ignore_index=True).drop_duplicates()
        merged.to_parquet(path)
        print(f"{name} update complete: {len(merged)} records")

patch_financial_data(all_stocks)

## 8. Download Price Adjustment Factors

Download the `adj_factor` series used to convert raw close prices into post-rights-adjusted (hfq) prices in the data cleaning stage.

In [ ]:
def download_adj_factor(stock_list, start_date='20150101', end_date='20241231'):
    save_dir = os.path.join(DATA_RAW, 'adj_factor')
    os.makedirs(save_dir, exist_ok=True)
    failed, skipped = [], 0

    for i, ts_code in enumerate(stock_list):
        save_path = os.path.join(save_dir, f"{ts_code}.parquet")
        if os.path.exists(save_path):
            skipped += 1
            continue
        df = None
        for attempt in range(3):
            try:
                df = pro.adj_factor(ts_code=ts_code,
                                    start_date=start_date, end_date=end_date)
                break
            except Exception:
                time.sleep((attempt + 1) * 2)
        if df is not None and len(df) > 0:
            df['trade_date'] = pd.to_datetime(df['trade_date'])
            df.sort_values('trade_date').to_parquet(save_path)
        else:
            failed.append(ts_code)
        if (i + 1) % 50 == 0:
            print(f"Progress: {i+1}/{len(stock_list)}, skipped {skipped}, failed {len(failed)}")
        time.sleep(0.5)

    print(f"Done: succeeded {len(stock_list)-len(failed)-skipped}, skipped {skipped}, failed {len(failed)}")
    return failed

failed_adj = download_adj_factor(all_stocks)

## 9. Data Completeness Check

Verify that all expected output files exist before proceeding to Chapter 02 (data cleaning).

In [ ]:
import os, sys
sys.path.append(r"D:\Quant\Multi_factor_project")
from config import *
import pandas as pd

# Check all data files
checks = {
    'daily_panel': os.path.join(DATA_CLEAN, 'daily_panel.parquet'),
    'income': os.path.join(DATA_RAW, 'financial', 'income.parquet'),
    'balancesheet': os.path.join(DATA_RAW, 'financial', 'balancesheet.parquet'),
    'cashflow': os.path.join(DATA_RAW, 'financial', 'cashflow.parquet'),
    'daily_basic': os.path.join(DATA_RAW, 'financial', 'daily_basic.parquet'),
    'hs300_members': os.path.join(DATA_RAW, 'industry', 'hs300_monthly_universe.parquet'),
    'hs300_index': os.path.join(DATA_RAW, 'daily', 'hs300_index.parquet'),
}

for name, path in checks.items():
    exists = os.path.exists(path)
    if exists:
        size = os.path.getsize(path) / 1024 / 1024
        print(f"OK {name}: {size:.1f} MB")
    else:
        print(f"MISSING {name}: not found, needs to be downloaded")

## 10. Merge Daily Price Panel

Concatenate all per-stock daily price files into a single long-format panel, sorted by ticker and date, and persist it to the clean data folder.

In [ ]:
import os, sys
sys.path.append(r"D:\Quant\Multi_factor_project")
from config import *
import pandas as pd

def merge_all_daily():
    save_dir = os.path.join(DATA_RAW, 'daily')
    
    # Read all single-stock files, excluding the index file
    files = [f for f in os.listdir(save_dir) 
             if f.endswith('.parquet') and f != 'hs300_index.parquet']
    
    print(f"Preparing to merge {len(files)} files...")
    
    all_df = []
    for i, f in enumerate(files):
        df = pd.read_parquet(os.path.join(save_dir, f))
        all_df.append(df)
        if (i+1) % 50 == 0:
            print(f"Read progress: {i+1}/{len(files)}")
    
    panel = pd.concat(all_df, ignore_index=True)
    panel['trade_date'] = pd.to_datetime(panel['trade_date'])
    panel = panel.sort_values(['ts_code', 'trade_date']).reset_index(drop=True)
    
    # Save to the clean data folder
    save_path = os.path.join(DATA_CLEAN, 'daily_panel.parquet')
    panel.to_parquet(save_path)
    
    print(f"\nMerge complete:")
    print(f"  Number of stocks: {panel['ts_code'].nunique()}")
    print(f"  Number of trading days: {panel['trade_date'].nunique()}")
    print(f"  Date range: {panel['trade_date'].min()} to {panel['trade_date'].max()}")
    print(f"  Total rows: {len(panel):,}")
    
    return panel

panel = merge_all_daily()